# 2.4 — Méthode apprise

1. Préparer les données et entraîner sur TRAIN.
2. Choisir la configuration sur VALIDATION.
3. Reconstruire les mentions comme pour le dictionnaire.
4. Comparer les deux méthodes, les figer, prédire sur TEST.
5. Expliquer cinq erreurs.

**Modèle** : BioBERT (`dmis-lab/biobert-base-cased-v1.1`), repris du TD NER. Code : `src/apprise.py`.


Les résultats sont écrits dans `resultats/extraction/` et le modèle dans `modeles/` (≈ 420 Mo, non joint au rendu).

In [1]:
import os, sys, json, time, shutil
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RACINE = Path("/content/drive/MyDrive/projet_nlp_sante")
    DOSSIER_CHECKPOINTS = Path("/content/checkpoints")   # disque local de Colab, pas le Drive
else:
    RACINE = Path.cwd().parent
    DOSSIER_CHECKPOINTS = RACINE / "modeles" / "checkpoints"
sys.path.insert(0, str(RACINE))

# Mode rapide : vérifie le parcours sur quelques documents (CPU), sans viser une performance.
MODE_RAPIDE = os.environ.get("PROJET_MODE_RAPIDE") == "1"
print("Colab :", EN_COLAB, "| racine :", RACINE, "| mode rapide :", MODE_RAPIDE)

Mounted at /content/drive
Colab : True | racine : /content/drive/MyDrive/projet_nlp_sante | mode rapide : False


In [2]:
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import (BertTokenizer, AutoModelForTokenClassification, BertForTokenClassification,
                          TrainingArguments,
                          DataCollatorForTokenClassification, set_seed)

from tals import erreurs
from src.cdr import charger_jsonl, mentions_reference
from src.apprise import (LABELS, LABEL2ID, POLITIQUE_DECODAGE, construire_fenetres, encoder, predire,
                         WeightedLossTrainer)
from src.extraction import tableau_scores, ecrire_predictions, lire_predictions

SEED = 42
MODELE = "dmis-lab/biobert-base-cased-v1.1"
# Paramètres de départ : ceux du TD (Partie3_Entrainement_et_Eval, MODE_RAPIDE=False)
PARAMS_TD = {"learning_rate": 2e-5, "batch_size": 8, "weight_decay": 0.01, "epochs": 3,
             "max_length": 512, "entity_weight": 10.0}
# Configurations comparées sur VALIDATION : poids de la perte (TD : 10 ; perte non pondérée : 1),
# et nombre d'époques (1, 2 ou 3, évalué à la fin de chaque époque).
POIDS_COMPARES = [10.0, 1.0]

RESULTATS = RACINE / "resultats" / "extraction"
SORTIE = RESULTATS / "essai_rapide" if MODE_RAPIDE else RESULTATS
SORTIE.mkdir(parents=True, exist_ok=True)
if MODE_RAPIDE:
    PARAMS_TD.update(epochs=1, max_length=128)
    POIDS_COMPARES = [10.0]

set_seed(SEED)
appareil = "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("Appareil :", appareil, "| torch", torch.__version__, "| transformers", transformers.__version__)

Appareil : GPU Tesla T4 | torch 2.11.0+cu130 | transformers 5.18.0


## Étape 1 — Préparer les données

On reprend les étiquettes BIO du 2.2. BioBERT découpe les mots en morceaux plus petits (sous-mots) : on vérifie que chaque mot garde la bonne étiquette. Les résumés sont découpés en groupes de phrases entières (512 sous-mots au maximum) plutôt que coupés n'importe où.

In [3]:
documents = charger_jsonl(RACINE / "data" / "corpus_extraction.jsonl")
bio = [json.loads(l) for l in open(RACINE / "data" / "bio_extraction.jsonl", encoding="utf-8")]
textes = {d.document_id: d.texte for d in documents}
bio_par_doc = {b["document_id"]: b for b in bio}
parts = {p: [d for d in documents if d.partition == p] for p in ["train", "validation", "test"]}
if MODE_RAPIDE:
    parts = {"train": parts["train"][:20], "validation": parts["validation"][:10], "test": parts["test"][:10]}

tokenizer = BertTokenizer.from_pretrained(MODELE)
fenetres = {p: construire_fenetres([bio_par_doc[d.document_id] for d in docs], textes, tokenizer,
                                   PARAMS_TD["max_length"])
            for p, docs in parts.items()}
donnees, tronques = {}, {}
for p in ["train", "validation"]:
    donnees[p], tronques[p] = encoder(fenetres[p], bio_par_doc, tokenizer, PARAMS_TD["max_length"])
pd.DataFrame({p: {"documents": len(parts[p]), "fenêtres": len(fenetres[p]),
                  "fenêtres tronquées": tronques.get(p, "—")} for p in parts})

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

,train,validation,test
documents,500,500,500
fenêtres,560,552,563
fenêtres tronquées,0,0,—


In [4]:
# Vérification de l'alignement sur une fenêtre : seul le premier sous-mot de chaque mot porte l'étiquette
ex = donnees["train"][0]
sous_mots = tokenizer.convert_ids_to_tokens(ex["input_ids"])
alignement = pd.DataFrame({"sous-mot": sous_mots,
                           "étiquette": ["·" if l == -100 else LABELS[l] for l in ex["labels"]]})
print(alignement.head(40).T.to_string())

# Contrôle global : autant d'étiquettes supervisées que de mots, et mêmes étiquettes que le BIO par mot
for p in ["train", "validation"]:
    for f, e in zip(fenetres[p], donnees[p]):
        etiquettes = [LABELS[l] for l in e["labels"] if l != -100]
        attendues = [bio_par_doc[f["document_id"]]["tags"][i] for i in f["mots"]]
        assert etiquettes == attendues, f["document_id"]
print("Alignement vérifié sur toutes les fenêtres de TRAIN et VALIDATION.")

              0           1     2      3    4        5    6    7     8     9       10     11     12      13  14          15     16    17     18 19  20  21     22    23    24       25 26           27    28         29      30         31    32   33        34  35     36        37   38     39
sous-mot   [CLS]          na  ##lo  ##xon  ##e  reverse  ##s  the  anti  ##hy  ##pert  ##ens  ##ive  effect  of           c  ##lon  ##id  ##ine  .  in  un  ##ane  ##st  ##he  ##tized  ,  spontaneous  ##ly          h  ##yper  ##tensive  rats  the  decrease  in  blood  pressure  and  heart
étiquette      ·  B-Chemical     ·      ·    ·        O    ·    O     O     ·       ·      ·      ·       O   O  B-Chemical      ·     ·      ·  O   O   O      ·     ·     ·        ·  O            O     ·  B-Disease       ·          ·     O    O         O   O      O         O    O      O
Alignement vérifié sur toutes les fenêtres de TRAIN et VALIDATION.


## Étape 2 — Entraîner et choisir la configuration

Paramètres du TD : taux d'apprentissage 2e-5, lots de 8, 3 époques, poids 10 sur les étiquettes B/I. On compare deux poids (10 et 1) et le modèle après 1, 2 et 3 époques. Critère : F1 sur les mentions de VALIDATION.

In [5]:
ref_val = mentions_reference(parts["validation"])
collator = DataCollatorForTokenClassification(tokenizer)

def entrainer(poids):
    set_seed(SEED)
    # Le config.json de BioBERT n'a pas de clé `model_type` : on indique la classe BERT explicitement.
    modele = BertForTokenClassification.from_pretrained(
        MODELE, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=LABEL2ID)
    dossier = DOSSIER_CHECKPOINTS / f"poids_{poids:g}"
    args = TrainingArguments(
        output_dir=str(dossier), save_strategy="epoch", eval_strategy="no", logging_strategy="epoch",
        learning_rate=PARAMS_TD["learning_rate"], per_device_train_batch_size=PARAMS_TD["batch_size"],
        num_train_epochs=PARAMS_TD["epochs"], weight_decay=PARAMS_TD["weight_decay"],
        fp16=torch.cuda.is_available(), report_to="none", seed=SEED, data_seed=SEED, save_total_limit=None)
    trainer = WeightedLossTrainer(model=modele, args=args, entity_weight=poids, train_dataset=donnees["train"],
                                  processing_class=tokenizer, data_collator=collator)
    t0 = time.perf_counter()
    trainer.train()
    duree = time.perf_counter() - t0
    points = sorted(dossier.glob("checkpoint-*"), key=lambda c: int(c.name.split("-")[1]))
    return points, duree, trainer.state.log_history

reglage, durees, historiques = [], {}, {}
for poids in POIDS_COMPARES:
    points, duree, historique = entrainer(poids)
    durees[poids], historiques[poids] = duree, historique
    for epoque, point in enumerate(points, start=1):
        modele = AutoModelForTokenClassification.from_pretrained(point).to("cuda" if torch.cuda.is_available() else "cpu")
        t0 = time.perf_counter()
        pred = predire(modele, tokenizer, fenetres["validation"], bio_par_doc, PARAMS_TD["max_length"])
        t_pred = time.perf_counter() - t0
        s = tableau_scores(ref_val, pred, "apprise", "validation").set_index("categorie")
        reglage.append({"poids": poids, "epoques": epoque, "checkpoint": str(point),
                        "P": s.loc["global (micro)", "precision"], "R": s.loc["global (micro)", "rappel"],
                        "F1": s.loc["global (micro)", "F1"], "F1 Chemical": s.loc["Chemical", "F1"],
                        "F1 Disease": s.loc["Disease", "F1"], "temps_entrainement_s": round(duree, 1),
                        "temps_prediction_validation_s": round(t_pred, 1)})
        del modele
    print(f"poids {poids:g} : entraînement {duree / 60:.1f} min")

reglage = pd.DataFrame(reglage)
reglage.drop(columns="checkpoint").to_csv(SORTIE / "apprise_reglage_validation.csv", index=False)
reglage.drop(columns="checkpoint").sort_values("F1", ascending=False)

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

... (16 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

poids 1 : entraînement 1.4 min


,poids,epoques,P,R,F1,F1 Chemical,F1 Disease,temps_entrainement_s,temps_prediction_validation_s
4,1.0,2,0.8533,0.8558,0.8546,0.9128,0.7813,84.6,17.2
5,1.0,3,0.8263,0.8826,0.8535,0.9104,0.7842,84.6,17.3
1,10.0,2,0.7182,0.8956,0.7971,0.8665,0.7168,81.9,16.0
2,10.0,3,0.6852,0.8855,0.7726,0.8499,0.6858,81.9,17.7
3,1.0,1,0.6849,0.8004,0.7382,0.8157,0.6480,84.6,16.8
0,10.0,1,0.5879,0.8624,0.6992,0.7894,0.5983,81.9,15.9


## Étape 3 — Reconstruire les mentions

Les étiquettes prédites sont regroupées en mentions, avec leurs positions dans le document. Règle fixée à l'avance : un `I-X` sans début avant lui ouvre une nouvelle mention. Mêmes catégories, mêmes positions et même calcul de score que pour le dictionnaire.

On garde la configuration au meilleur F1 sur VALIDATION (à égalité, la moins coûteuse).

In [6]:
retenue = reglage.sort_values(["F1", "epoques"], ascending=[False, True]).iloc[0]
print(f"Configuration retenue : poids {retenue.poids:g}, {retenue.epoques} époque(s) — F1 VALIDATION {retenue.F1}")

DOSSIER_MODELE = RACINE / "modeles" / ("biobert_essai_rapide" if MODE_RAPIDE else "biobert_retenu")
if DOSSIER_MODELE.exists():
    shutil.rmtree(DOSSIER_MODELE)
shutil.copytree(retenue.checkpoint, DOSSIER_MODELE)
modele = AutoModelForTokenClassification.from_pretrained(DOSSIER_MODELE).to("cuda" if torch.cuda.is_available() else "cpu")

t0 = time.perf_counter()
pred_val = predire(modele, tokenizer, fenetres["validation"], bio_par_doc, PARAMS_TD["max_length"])
t_val = time.perf_counter() - t0
scores_val = tableau_scores(ref_val, pred_val, "apprise", "validation", t_val)
scores_val.to_csv(SORTIE / "scores_apprise_validation.csv", index=False)
ecrire_predictions(pred_val, "apprise", parts["validation"], SORTIE / "predictions_apprise_validation.jsonl")
scores_val

Configuration retenue : poids 1, 2 époque(s) — F1 VALIDATION 0.8546


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,methode,partition,categorie,VP,FP,FN,precision,rappel,F1,denominateur_nul,temps_s
0,apprise,validation,Chemical,4886,473,461,0.9117,0.9138,0.9128,False,NaN
1,apprise,validation,Disease,3322,938,922,0.7798,0.7828,0.7813,False,NaN
2,apprise,validation,global (micro),8208,1411,1383,0.8533,0.8558,0.8546,False,17.96


## Étape 4 — Comparer les deux méthodes sur VALIDATION

Mêmes documents, même référence, même score. Les prédictions du dictionnaire viennent du 2.3.

In [7]:
docs_val = {d.document_id for d in parts["validation"]}
pred_dico = {m for m in lire_predictions(RESULTATS / "predictions_dictionnaire_validation.jsonl") if m.doc in docs_val}
temps_dico = json.loads((RESULTATS / "dictionnaire_config_gelee.json").read_text())["temps_s"]["extraction_validation"]
comparaison = pd.concat([tableau_scores(ref_val, pred_dico, "dictionnaire", "validation", temps_dico),
                         scores_val], ignore_index=True)
comparaison.to_csv(SORTIE / "comparaison_validation.csv", index=False)
comparaison

,methode,partition,categorie,VP,FP,FN,precision,rappel,F1,denominateur_nul,temps_s
0,dictionnaire,validation,Chemical,3501,228,1846,0.9389,0.6548,0.7715,False,NaN
1,dictionnaire,validation,Disease,2674,586,1570,0.8202,0.6301,0.7127,False,NaN
2,dictionnaire,validation,global (micro),6175,814,3416,0.8835,0.6438,0.7449,False,1.39
3,apprise,validation,Chemical,4886,473,461,0.9117,0.9138,0.9128,False,NaN
4,apprise,validation,Disease,3322,938,922,0.7798,0.7828,0.7813,False,NaN
5,apprise,validation,global (micro),8208,1411,1383,0.8533,0.8558,0.8546,False,17.96


## Gel et prédictions sur TEST

Les deux méthodes sont figées et appliquées aux textes de TEST. Les scores TEST sont calculés dans `5_evaluation.ipynb`, sans nouveau réglage.

In [8]:
t0 = time.perf_counter()
pred_test = predire(modele, tokenizer, fenetres["test"], bio_par_doc, PARAMS_TD["max_length"])
t_test = time.perf_counter() - t0
ecrire_predictions(pred_test, "apprise", parts["test"], SORTIE / "predictions_apprise_test.jsonl")

gel = {"modele": MODELE, "parametres_td": PARAMS_TD, "seed": SEED,
       "config_retenue": {"entity_weight": float(retenue.poids), "epochs": int(retenue.epoques)},
       "politique_decodage": POLITIQUE_DECODAGE, "fenetres": "phrases entières, max_length sous-mots",
       "alignement_sous_mots": "premier sous-mot supervisé (tals.aligner_sous_mots)",
       "appareil": appareil, "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
       "temps_s": {"entrainement": round(durees[retenue.poids], 1), "prediction_validation": round(t_val, 1),
                   "prediction_test": round(t_test, 1)},
       "fenetres_par_partition": {p: len(f) for p, f in fenetres.items()},
       "predictions_test": len(pred_test), "mode_rapide": MODE_RAPIDE}
(SORTIE / "apprise_config_gelee.json").write_text(json.dumps(gel, ensure_ascii=False, indent=2), encoding="utf-8")
json.dump(historiques, open(SORTIE / "apprise_historique_entrainement.json", "w"), indent=2, default=str)

if not MODE_RAPIDE:
    manifest = json.loads((RACINE / "manifest.json").read_text(encoding="utf-8"))
    manifest["methode_apprise"] = {
        "type_modele": "BERT ajusté en classification de tokens BIO (src/apprise.py, repris du TD NER)",
        "checkpoint_preentraine": MODELE,
        "hyperparametres": {"learning_rate": PARAMS_TD["learning_rate"], "batch_size": PARAMS_TD["batch_size"],
                            "epochs": int(retenue.epoques), "weight_decay": PARAMS_TD["weight_decay"],
                            "max_length": PARAMS_TD["max_length"], "entity_weight": float(retenue.poids),
                            "seed": SEED},
        "config_retenue_sur_validation": f"poids {retenue.poids:g}, {retenue.epoques} époque(s), F1 {retenue.F1}",
        "fichiers": ["resultats/extraction/apprise_config_gelee.json",
                     "resultats/extraction/apprise_reglage_validation.csv"]}
    (RACINE / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(gel, ensure_ascii=False, indent=2))

{
  "modele": "dmis-lab/biobert-base-cased-v1.1",
  "parametres_td": {
    "learning_rate": 2e-05,
    "batch_size": 8,
    "weight_decay": 0.01,
    "epochs": 3,
    "max_length": 512,
    "entity_weight": 10.0
  },
  "seed": 42,
  "config_retenue": {
    "entity_weight": 1.0,
    "epochs": 2
  },
  "politique_decodage": "reparer",
  "fenetres": "phrases entières, max_length sous-mots",
  "alignement_sous_mots": "premier sous-mot supervisé (tals.aligner_sous_mots)",
  "appareil": "GPU Tesla T4",
  "versions": {
    "torch": "2.11.0+cu130",
    "transformers": "5.18.0"
  },
  "temps_s": {
    "entrainement": 84.6,
    "prediction_validation": 18.0,
    "prediction_test": 17.7
  },
  "fenetres_par_partition": {
    "train": 560,
    "validation": 552,
    "test": 563
  },
  "predictions_test": 9886,
  "mode_rapide": false
}


## Étape 5 — Erreurs d'extraction (VALIDATION)

Types d'erreurs : **frontière**, **type**, **manquée**, **parasite** (voir 2.3). On repère les mentions absentes de TRAIN et on compare avec le dictionnaire. Les cinq erreurs commentées sont dans `resultats/extraction/erreurs_commentees.md`.

In [9]:
def surface(m):
    return textes[m.doc][m.debut:m.fin]

formes_train = {surface_.lower() for d in parts["train"] for surface_ in (x.texte for x in d.mentions)}
err = erreurs(ref_val, pred_val)
print({k: len(v) for k, v in err.items()})
manquees = err["manquee"]
print(f"Manquées dont la forme est absente de TRAIN : {sum(surface(m).lower() not in formes_train for m in manquees)} / {len(manquees)}")

exemples = []
for att, obt in err["frontiere"][:5]:
    exemples.append({"erreur": "frontière", "référence": f"{surface(att)} <{att.type}>", "prédit": f"{surface(obt)} <{obt.type}>", "PMID": att.doc})
for att, obt in err["type"][:5]:
    exemples.append({"erreur": "type", "référence": f"{surface(att)} <{att.type}>", "prédit": f"{surface(obt)} <{obt.type}>", "PMID": att.doc})
for m in manquees[:5]:
    exemples.append({"erreur": "manquée" + (" (forme inconnue)" if surface(m).lower() not in formes_train else ""),
                     "référence": f"{surface(m)} <{m.type}>", "prédit": "—", "PMID": m.doc})
for m in err["parasite"][:5]:
    exemples.append({"erreur": "parasite", "référence": "—", "prédit": f"{surface(m)} <{m.type}>", "PMID": m.doc})
pd.DataFrame(exemples)

{'correcte': 8208, 'frontiere': 631, 'type': 11, 'manquee': 741, 'parasite': 769}
Manquées dont la forme est absente de TRAIN : 330 / 741


,erreur,référence,prédit,PMID
0,frontière,secondary hyperparathyroidism <Disease>,hyperparathyroidism <Disease>,10027919
1,frontière,secondary hyperparathyroidism <Disease>,hyperparathyroidism <Disease>,10027919
2,frontière,secondary hyperparathyroidism <Disease>,hyperparathyroidism <Disease>,10027919
3,frontière,ischemic <Disease>,ischemic cardiac disease <Disease>,10074612
4,frontière,renal tubular dysfunction <Disease>,dysfunction <Disease>,10193809
5,type,SIADH <Disease>,SIADH <Chemical>,11195262
6,type,hepatitis B <Disease>,hepatitis B <Chemical>,15867025
7,type,hepatitis B <Disease>,hepatitis B <Chemical>,15867025
8,type,rubella <Disease>,rubella <Chemical>,15867025
9,type,Hepatitis B <Disease>,Hepatitis B <Chemical>,15867025


In [10]:
# Croisement avec la méthode par règles : qui trouve quoi ?
seul_apprise = (pred_val & ref_val) - pred_dico
seul_dico = (pred_dico & ref_val) - pred_val
print(f"Correctes seulement pour la méthode apprise : {len(seul_apprise)} ; seulement pour le dictionnaire : {len(seul_dico)}")
print("Exemples apprise seule :", sorted({surface(m) for m in seul_apprise})[:15])
print("Exemples dictionnaire seul :", sorted({surface(m) for m in seul_dico})[:15])

Correctes seulement pour la méthode apprise : 2616 ; seulement pour le dictionnaire : 583
Exemples apprise seule : ["0,0'-diisopropyl phosphorofluoridate", '1,25-dihydroxyvitamin D', '1-(2-chloroethyl)-3-cyclohexyl-1-nitrosourea', '1-(2-phenethyl)-4-(2-pyridyl)-piperazine', '1-anisoyl-2-pyrrolidinone', '1-methyl-4-phenyl-1,2,3,6-tetrahydropyridine', '11-ketopregnenolone sulphate', '13-cis-retinoic acid', '14C-41 099', '15-F(2t)-IsoP', '15-F(2t)-isoprostane', '2,3,5-triphenyltetrazolium', '2-bromoethylamine', '22-oxacalcitriol', '2R,4R-APDC']
Exemples dictionnaire seul : ['4-DAMP', '5-HT', 'ADP', 'AIDS', 'AMP', 'ATP', 'Acute liver failure', 'Acute renal failure', 'CHF', 'CM', 'CO2', 'Ca', 'Cardiotoxic', 'Cholesterol', 'DA']
